# Short notebook 1: Kaggle submission v16 (public score 0.99513)

**TDT4173 Modern Machine Learning in Practice, course project, group 180**

* Kaggle team name: [180] the strugglers
* Arghya Mukherjee, student ID 180026
* Jonatan Halperin, student ID 170762
* Hemal Sandeepa Gunasinghe, student ID 173254

This notebook contains only the steps needed to reproduce `v16` from the raw data:

1. Load the raw data and build the features of each (case, unit, hour).
2. System features: the state of all 17 reservoirs.
3. A simplified copy of SHOP's weekly problem (a linear program) for every case; its solution becomes features.
4. One XGBoost model per unit, trained on all labelled cases, average of 3 seeds.
5. A neural network with system features (3 seeds), rank-blended 0.95 trees / 0.05 network.

How the settings were chosen (validation on three periods) is described in the report.

**Expected files:** `data/kernel/*.csv`, `data/extended/Constraint_min_flow.csv`, `data/extended/Constraint_min_volume.csv`,
`data/extended/Tokke_Vinje_topology.yaml` and `prediction_mapping.csv` (paths can be changed in the first code cell).
Packages: see `requirements.txt` (Python 3.13). On macOS, XGBoost also needs the OpenMP library (`brew install libomp`).

**Run time** on a laptop (Apple M-series, CPU only): about 1–1.5 hours. Peak memory under 8 GB. (Tested end to end on a 2-core Linux machine: 86 minutes for Short notebook 2.)

In [ ]:
import sys
import time

import numpy as np
import pandas as pd
import xgboost as xgb
import yaml
from scipy import sparse
from scipy.optimize import linprog
from scipy.stats import rankdata

DATA = "data"                                   # folder with kernel/ and extended/
MAPPING = "prediction_mapping.csv"              # test cases (Run No, starttime)
OUTPUT = "submission_v16.csv"
np.random.seed(42)

## Data and features

In [ ]:
# Raw data. All timestamps are UTC; the time zone is dropped so everything can be indexed together.
def read(path, index_col="date"):
    df = pd.read_csv(path, parse_dates=[index_col], index_col=index_col)
    if df.index.tz is not None:
        df.index = df.index.tz_convert(None)
    return df

water_value = read(f"{DATA}/kernel/Synthetic_water_value_2015_2024.csv")           # EUR/MWh, daily
volume = read(f"{DATA}/kernel/Historical_volume_2015_2024.csv")                     # Mm³, daily
day_ahead_price = read(f"{DATA}/kernel/Historical_day_ahead_price_2015_2025.csv")
day_ahead_price = day_ahead_price.rename(columns={"Day-ahead price (EUR/MWh)": "price"}).resample("h").mean()
inflow = read(f"{DATA}/kernel/Historical_inflow_1958_2025.csv")                      # m³/s, hourly
unit_commitment_decisions = read(f"{DATA}/kernel/Unit_commitment_decisions.csv", "starttime")
min_flow = read(f"{DATA}/extended/Constraint_min_flow.csv")                          # m³/s, hourly
min_volume = read(f"{DATA}/extended/Constraint_min_volume.csv")                      # Mm³, hourly
test_cases = pd.read_csv(MAPPING, parse_dates=["starttime"])

with open(f"{DATA}/extended/Tokke_Vinje_topology.yaml") as f:
    topology = yaml.safe_load(f)
max_vol = pd.Series({name: r["max_vol"] for name, r in topology["model"]["reservoir"].items()})
fill = volume / max_vol                                                              # share of max volume

# max flow (m³/s) of each unit = largest flow in its turbine efficiency curves; plant = sum of its units
unit_max_flow = {gen: max(max(curve["x"]) for curve in g["turb_eff_curves"])
                 for gen, g in topology["model"]["generator"].items()}
plant_max_flow = {}
for gen, q in unit_max_flow.items():
    plant_max_flow[gen.rsplit("_G", 1)[0]] = plant_max_flow.get(gen.rsplit("_G", 1)[0], 0) + q

# labels: 14 units x 168 hours per case; columns ordered unit by unit
label_cols = [c for c in unit_commitment_decisions.columns if c.startswith("result_committed")]
units = list(dict.fromkeys(c.replace("result_committed_", "").rsplit("_t", 1)[0] for c in label_cols))
Y = unit_commitment_decisions[label_cols].to_numpy().reshape(len(unit_commitment_decisions), len(units), 168)
print(len(unit_commitment_decisions), "training cases,", len(test_cases), "test cases,", len(units), "units")

In [ ]:
# Which reservoir(s) each plant draws from, where it releases its water, and which plants are upstream
plant_sources = {}
for conn in topology["connections"]:
    if conn["to_type"] == "plant" and conn["from_type"] in ("reservoir", "tunnel"):
        plant_sources.setdefault(conn["to"], []).append(conn["from"])
tunnel_sources = {conn["to"]: conn["from"] for conn in topology["connections"]
                  if conn["to_type"] == "tunnel" and conn["from_type"] == "reservoir"}
plant_reservoir = {plant: [tunnel_sources.get(s, s) for s in sources] for plant, sources in plant_sources.items()}

plant_downstream = {}
for conn in topology["connections"]:
    if conn["from_type"] == "plant" and conn["to_type"] == "reservoir":
        plant_downstream[conn["from"]] = [conn["to"]]
plant_downstream["Lio"] = ["Bandak"]   # Lio -> river r_Vest_Vassdraget -> Bandak
plant_downstream["Hogga"] = []         # Hogga is at the bottom: the water leaves the system

plant_upstream = {plant: [other for other, down in plant_downstream.items() if set(down) & set(plant_reservoir[plant])]
                  for plant in plant_reservoir}
plant_upstream["Vinje"] = ["Songa"]   # Songa -> Totak -> tunnel -> Vaamarvatn

# "Normal" inflow for each day of the year (1958–2014) and small helpers
inflow_daily = inflow.resample("D").mean()
inflow_normal = inflow_daily.loc[:"2014"].groupby(inflow_daily.loc[:"2014"].index.dayofyear).mean()

def hourly_matrix(series, starts):
    """168 hourly values from each start date -> shape (n_cases, 168)."""
    return np.array([series.loc[s : s + pd.Timedelta(hours=167)].to_numpy() for s in starts])

def rolling_within_case(M, window):
    """Centred rolling mean along the hours of each case."""
    return pd.DataFrame(M).T.rolling(window, center=True, min_periods=1).mean().T.to_numpy()

def daily_value(df, dates, columns):
    """Mean of the given reservoir columns on the given dates (0 if there are no reservoirs)."""
    if len(columns) == 0:
        return np.zeros(len(dates))
    return df.loc[dates, columns].mean(axis=1).to_numpy()

In [ ]:
# Features of each (case, unit, hour); the groups A–D are described in the report
def build_features(starts):
    """One row per (case, unit, hour). Works for both train and test start dates."""
    starts = pd.DatetimeIndex(starts)
    ends = starts + pd.Timedelta(days=7)
    n_cases, n_hours = len(starts), 168
    rep = lambda v: np.repeat(v, n_hours)            # one value per case -> all 168 hours
    doy = np.minimum(starts.dayofyear, 365)

    # ---------- features that are the same for all units ----------
    P = hourly_matrix(day_ahead_price["price"], starts)
    desc_rank = n_hours - P.argsort(axis=1).argsort(axis=1)   # 1 = most expensive hour of the week, 168 = cheapest
    week_mean = P.mean(axis=1, keepdims=True)
    week_std = P.std(axis=1, keepdims=True) + 1e-6
    total_inflow = hourly_matrix(inflow.sum(axis=1), starts)
    total_normal = inflow_normal.sum(axis=1).loc[doy].to_numpy()
    P_after = hourly_matrix(day_ahead_price["price"], starts + pd.Timedelta(hours=168))[:, :48]   # 48 h after the week
    after24, after48 = P_after[:, :24].mean(axis=1), P_after[:, 24:].mean(axis=1)
    vest_min = hourly_matrix(min_flow["r_Vest_Vassdraget"], starts)          # required minimum flow (m³/s)
    vest_deficit = np.clip(vest_min - hourly_matrix(inflow["r_Vest_Vassdraget"], starts), 0, None)   # not covered by natural inflow

    shared = {
        "hour": np.tile(np.arange(n_hours), n_cases),
        "hour_of_day": np.tile(np.arange(n_hours) % 24, n_cases),
        "month": rep(starts.month.to_numpy()),
        "price": P.ravel(),
        "price_rank": (P.argsort(axis=1).argsort(axis=1) / (n_hours - 1)).ravel(),
        "price_z": ((P - week_mean) / week_std).ravel(),
        "price_week_mean": rep(week_mean[:, 0]),
        "price_roll6": rolling_within_case(P, 7).ravel(),
        "price_roll24": rolling_within_case(P, 25).ravel(),
        "inflow_total_week": rep(total_inflow.mean(axis=1)),
        "inflow_total_anom": rep(total_inflow.mean(axis=1) / total_normal),
        # --- group A: exact date + system-wide constraints ---
        "doy": rep(doy),
        "con_totak_minvol": (hourly_matrix(min_volume["Totak"], starts) / max_vol["Totak"]).ravel(),
        "con_staavatn_minvol": (hourly_matrix(min_volume["Staavatn"], starts) / max_vol["Staavatn"]).ravel(),
        "con_vest_minflow": vest_min.ravel(),
        # --- group D: minimum flow in Vest-Vassdraget not met by natural inflow (Lio must release water) ---
        "vest_flow_deficit": vest_deficit.ravel(),
        "vest_deficit_share": rep((vest_deficit > 0).mean(axis=1)),
    }

    # price minus end water value for every plant (used for the cascade features)
    diff_end_by_plant = {pl: P - daily_value(water_value, ends, plant_reservoir[pl])[:, None]
                         for pl in plant_reservoir}

    # ---------- features that depend on the unit's reservoirs ----------
    blocks = []
    for unit in units:
        plant = unit.rsplit("_G", 1)[0]
        up = plant_reservoir[plant]          # where the water comes from
        down = plant_downstream[plant]       # where the water goes

        wv_start = daily_value(water_value, starts, up)
        wv_end = daily_value(water_value, ends, up)                         # THE key quantity
        net_wv_end = wv_end - daily_value(water_value, ends, down)          # value lost by moving water down
        diff_end = P - wv_end[:, None]                                      # price minus end water value
        diff_net = P - net_wv_end[:, None]
        res_inflow = hourly_matrix(inflow[up].sum(axis=1), starts)
        res_normal = inflow_normal[up].sum(axis=1).loc[doy].to_numpy()

        # ---------- water budget ----------
        vol_start = volume.loc[starts, up].sum(axis=1).to_numpy()             # Mm³ in the reservoir(s) at start
        inflow_week = res_inflow.sum(axis=1) * 3600 / 1e6                     # Mm³ flowing in during the week
        free_space = max_vol[up].sum() - vol_start                            # Mm³ before the reservoir is full
        plant_per_hour = plant_max_flow[plant] * 3600 / 1e6                   # Mm³ the plant uses per hour at full load
        unit_per_hour = unit_max_flow[unit] * 3600 / 1e6

        hours_available = np.clip((vol_start + inflow_week) / plant_per_hour, 0, 168)
        hours_forced = np.clip((inflow_week - free_space) / plant_per_hour, 0, 168)
        unit_hours_available = np.clip((vol_start + inflow_week) / unit_per_hour, 0, 168)

        ups = plant_upstream[plant]
        if ups:
            up_diff = np.mean([diff_end_by_plant[q] for q in ups], axis=0)
            up_share = np.mean([(diff_end_by_plant[q] > 0).mean(axis=1) for q in ups], axis=0)
        else:
            up_diff = np.zeros_like(P)
            up_share = np.zeros(n_cases)

        # ---------- group A: constraints on the unit's own reservoir(s) ----------
        flow_cols = [f"b_{r}" for r in up if f"b_{r}" in min_flow.columns]   # minimum release from the reservoir
        vol_cols = [r for r in up if r in min_volume.columns]                # minimum volume of the reservoir
        minflow_out = hourly_matrix(min_flow[flow_cols].sum(axis=1), starts) if flow_cols else np.zeros_like(P)
        minvol_up = hourly_matrix(min_volume[vol_cols].sum(axis=1), starts) if vol_cols else np.zeros_like(P)
        minvol_headroom = np.clip((vol_start[:, None] - minvol_up) / plant_per_hour, -168, 1000)   # full-load hours above the minimum

        # ---------- group B: the water-value path through the week ----------
        W = np.stack([daily_value(water_value, starts + pd.Timedelta(days=d), up) for d in range(8)], axis=1)  # (cases, 8 days)
        wv_day = W[:, np.arange(n_hours) // 24]                                # water value of the hour's own day

        block = pd.DataFrame({
            "case": rep(np.arange(n_cases)),
            "unit": unit,
            **shared,
            "wv_start": rep(wv_start),
            "wv_end": rep(wv_end),
            "net_wv_end": rep(net_wv_end),
            "price_minus_wv_end": diff_end.ravel(),
            "ratio_price_wv_end": (P / (np.abs(wv_end[:, None]) + 1)).ravel(),
            "share_above_wv_end": rep((diff_end > 0).mean(axis=1)),
            "roll6_minus_wv_end": rolling_within_case(diff_end, 7).ravel(),
            "roll24_minus_wv_end": rolling_within_case(diff_end, 25).ravel(),
            "above_wv_end_run": rolling_within_case((diff_end > 0).astype(float), 13).ravel(),
            "price_minus_net_end": diff_net.ravel(),
            "share_above_net_end": rep((diff_net > 0).mean(axis=1)),
            "fill": rep(daily_value(fill, starts, up)),
            "fill_down": rep(daily_value(fill, starts, down)),
            "inflow_res": res_inflow.ravel(),
            "inflow_res_week": rep(res_inflow.mean(axis=1)),
            "inflow_res_anom": rep(res_inflow.mean(axis=1) / (res_normal + 1e-3)),
            "upstream_roll6_minus_wv_end": rolling_within_case(up_diff, 7).ravel(),
            "upstream_share_above": rep(up_share),
            "hours_available": rep(hours_available),
            "hours_forced": rep(hours_forced),
            "unit_hours_available": rep(unit_hours_available),
            "rank_minus_available": (desc_rank - hours_available[:, None]).ravel(),
            "rank_minus_forced": (desc_rank - hours_forced[:, None]).ravel(),
            # --- group A ---
            "minflow_out": minflow_out.ravel(),
            "minvol_up": (minvol_up / max_vol[up].sum()).ravel(),
            "minvol_headroom": minvol_headroom.ravel(),
            # --- group B ---
            "price_minus_wv_day": (P - wv_day).ravel(),
            "wv_day_rel": ((wv_day - wv_end[:, None]) / (np.abs(wv_end[:, None]) + 1)).ravel(),
            "wv_path_slope": rep((W[:, 7] - W[:, 0]) / (np.abs(W[:, 7]) + 1)),
            # --- group C: prices just after the week ---
            "after24_minus_wv_end": rep(after24 - wv_end),
            "after48_minus_wv_end": rep(after48 - wv_end),
            "after24_rel_week": rep(after24 / (np.abs(week_mean[:, 0]) + 1)),
        })
        numeric = block.columns.difference(["case", "unit"])
        block[numeric] = block[numeric].astype("float32")   # float32 per unit block: saves memory
        blocks.append(block)

    X = pd.concat(blocks, ignore_index=True)
    X["unit"] = pd.Categorical(X["unit"], categories=units)
    return X

In [ ]:
# Features for training and test cases; rows ordered (unit, case, hour)
t0 = time.time()
X = build_features(unit_commitment_decisions.index)
X_test = build_features(test_cases["starttime"])
y = Y.transpose(1, 0, 2).ravel()                                    # same order as X
print(X.shape, X_test.shape, f"{time.time() - t0:.0f}s")             # (6872544, 51) (1705200, 51)

absolute_levels = ["price", "price_week_mean", "price_roll6", "price_roll24", "wv_start", "wv_end"]
feat_A = ["doy", "con_totak_minvol", "con_staavatn_minvol", "con_vest_minflow",
          "minflow_out", "minvol_up", "minvol_headroom"]                        # constraints + exact date
feat_B = ["price_minus_wv_day", "wv_day_rel", "wv_path_slope"]                   # water-value path in the week
feat_C = ["after24_minus_wv_end", "after48_minus_wv_end", "after24_rel_week"]    # prices just after the week
feat_D = ["vest_flow_deficit", "vest_deficit_share"]                             # Lio's minimum-flow deficit
v6_features = [c for c in X.columns if c not in ["case"] + absolute_levels + feat_A + feat_B + feat_C + feat_D]
unit_feats = [f for f in v6_features + feat_A + feat_B + feat_C + feat_D if f != "unit"]   # 43; one model per unit
print(len(unit_feats), "unit features")

In [ ]:
# System features: the state of all 17 reservoirs, one row per (case, hour) = the order of each unit's rows in X
reservoirs = list(water_value.columns)

def system_features(starts):
    starts = pd.DatetimeIndex(starts)
    ends = starts + pd.Timedelta(days=7)
    P = hourly_matrix(day_ahead_price["price"], starts)
    wv_end = water_value.loc[ends, reservoirs].to_numpy()
    fill_start = fill.loc[starts, reservoirs].to_numpy()
    inflow_week = np.stack([hourly_matrix(inflow[r_], starts).mean(axis=1) for r_ in reservoirs], axis=1)
    rep = lambda v: np.repeat(v, 168)
    cols = {}
    for j, r_ in enumerate(reservoirs):
        cols[f"sys_margin_{r_}"] = (P - wv_end[:, j][:, None]).ravel()          # price minus end water value
    for j, r_ in enumerate(reservoirs):
        cols[f"sys_wv_end_{r_}"] = rep(wv_end[:, j])
        cols[f"sys_fill_{r_}"] = rep(fill_start[:, j])
        cols[f"sys_inflow_{r_}"] = rep(inflow_week[:, j])
    return pd.DataFrame(cols).astype("float32")

SYS = system_features(unit_commitment_decisions.index)
SYS_test = system_features(test_cases["starttime"])
n_cases = len(unit_commitment_decisions)
for u in units:       # every unit's block of rows in X is ordered (case, hour), like SYS
    m = (X["unit"] == u).to_numpy()
    assert (X.loc[m, "case"].to_numpy() == np.repeat(np.arange(n_cases), 168)).all()
print(SYS.shape, SYS_test.shape)   # (490896, 68) (121800, 68)

## Simulation: a simplified copy of SHOP's weekly problem (linear program)

For each case the LP maximises, over the 168 hours, (price − water value of the plant's source reservoir) × production,
minus the bypass/spill/river costs from the topology file and the value of water lost through bypasses and spillways
(valued along the whole path to the sea). Constraints: hourly water balance of all 17 reservoirs, volume limits,
plant capacities, and the minimum-flow / minimum-volume series (soft, high penalty). Simplifications: head fixed at the
start level, efficiency 0.88, no on/off or start costs, no time delays, free tunnels between reservoirs that share a
water value. Solved with HiGHS (`scipy.optimize.linprog`) at water-value scales 0.9, 1.0 and 1.1.
For each plant and hour it gives the planned flow share (`frac`) and the margin: price minus the LP's marginal value
of the water in EUR/MWh (`margin`, from the dual values). About 0.1 s per LP; 3,647 cases x 3 scales take 15–25 minutes.

In [ ]:
H = 168
ETA = 0.88              # turbine x generator efficiency (typical of the YAML curves)
M3S_H = 0.0036          # Mm³ moved by 1 m³/s during one hour
MINFLOW_PENALTY = 1e4   # EUR per m³/s-hour below a minimum flow
MINVOL_PENALTY = 1e6    # EUR per Mm³-hour below a minimum volume
SCALES = (0.9, 1.0, 1.1)

# Reservoirs share their water value with the plant they feed (equal columns in the water-value data)
GROUP_PLANT = {
    "Bandak": "Hogga", "Vinjevatn": "Tokke", "Byrtevatn": "Lio", "Botnedalsvatn": "Byrte",
    "Vaamarvatn": "Vinje", "Totak": "Vinje", "Venemo": "Vinje", "Hyljelihyl": "Vinje",
    "Vatjern": "Haukeli", "Langeidvatn": "Haukeli", "Songavatn": "Songa", "Bitdalsvatn": "Songa",
    "Foersvatn": "Kjela", "Langesae": "Kjela", "Bordalsvatn": "Kjela", "Kjelavatn": "Vesle Kjela", "Staavatn": "Vesle Kjela",
}
PLANTS = ["Hogga", "Tokke", "Lio", "Byrte", "Vinje", "Haukeli", "Songa", "Kjela", "Vesle Kjela"]
MINFLOW_RIVERS = ["b_Kjelavatn", "b_Hyljelihyl", "b_Vinjevatn", "b_Byrtevatn", "b_Bandak", "r_Vest_Vassdraget"]
MINVOL_RES = ["Totak", "Byrtevatn", "Staavatn"]
COLLECTOR = "r_Vest_Vassdraget"   # river without its own reservoir; everything in it flows to Bandak


def build_model(topo):
    """LP structure (the same for every case): flows, water-balance matrix, capacity and constraint rows."""
    m, conns = topo["model"], topo["connections"]
    res = list(m["reservoir"])
    ri = {r: i for i, r in enumerate(res)}

    def dest_of(node):   # reservoir that a river/plant/tunnel releases into (None = leaves the system)
        for c in conns:
            if c["from"] == node and c["to_type"] == "reservoir":
                return c["to"]
            if c["from"] == node and c["to_type"] == "river":
                return dest_of(c["to"])
        return None

    def into_collector(node):
        return any(c["from"] == node and c["to"] == COLLECTOR for c in conns)

    tunnel_src = {c["to"]: c["from"] for c in conns if c["to_type"] == "tunnel" and c["from_type"] == "reservoir"}
    edges = []
    plant_src = {p: [] for p in PLANTS}
    for c in conns:
        if c["to_type"] == "plant" and c["from_type"] in ("reservoir", "tunnel"):
            plant_src[c["to"]].append(tunnel_src.get(c["from"], c["from"]))
    for p in PLANTS:                                  # one flow per plant source (Songa has two)
        for s in plant_src[p]:
            edges.append(dict(kind="plant", src=s, dst=dest_of(p), cost=0.0, ub=None, plant=p, river=None,
                              collector=into_collector(p)))
    river_src = {c["to"]: c["from"] for c in conns if c["to_type"] == "river" and c["from_type"] == "reservoir"}
    for name, rv in m["river"].items():               # bypass gates (b_), spillways (s_), regulated rivers (r_)
        if name == COLLECTOR:
            continue
        src, dst, coll = river_src[name], dest_of(name), into_collector(name)
        if "flow_cost_curve" in rv:                   # two linear cost pieces: 1.6 up to 40 m³/s, then 3.3
            edges.append(dict(kind="river", src=src, dst=dst, cost=1.6, ub=40.0, plant=None, river=name, collector=coll))
            edges.append(dict(kind="river", src=src, dst=dst, cost=3.3, ub=960.0, plant=None, river=name, collector=coll))
        else:
            edges.append(dict(kind="spill" if name.startswith("s_") else "bypass", src=src, dst=dst,
                              cost=float(rv.get("flow_cost_const", 0.0)), ub=rv.get("max_flow_const"),
                              plant=None, river=name, collector=coll))
    for tname, s in tunnel_src.items():               # transfer tunnels: free, both directions, at most 500 m³/s
        d = dest_of(tname)
        if d is None:
            continue
        for a, b in [(s, d), (d, s)]:
            edges.append(dict(kind="tunnel", src=a, dst=b, cost=0.0, ub=500.0, plant=None, river=None, collector=False))

    E, R, P, F, MV = len(edges), len(res), len(PLANTS), len(MINFLOW_RIVERS), len(MINVOL_RES)
    o_v, o_f, o_m = E * H, E * H + R * H, E * H + R * H + F * H     # [flows | volumes | min-flow slack | min-vol slack]
    n = o_m + MV * H
    t = np.arange(H)

    rows, cols, vals = [], [], []                     # water balance: V[t] - V[t-1] - M3S_H*(in - out) = M3S_H*inflow
    for r in range(R):
        rows += [r * H + t, r * H + t[1:]]; cols += [o_v + r * H + t, o_v + r * H + t[:-1]]
        vals += [np.ones(H), -np.ones(H - 1)]
    for e, ed in enumerate(edges):
        if ed["src"] is not None:
            rows += [ri[ed["src"]] * H + t]; cols += [e * H + t]; vals += [np.full(H, M3S_H)]
        if ed["dst"] is not None:
            rows += [ri[ed["dst"]] * H + t]; cols += [e * H + t]; vals += [np.full(H, -M3S_H)]
    A_eq = sparse.csr_matrix((np.concatenate(vals), (np.concatenate(rows), np.concatenate(cols))), shape=(R * H, n))

    rows, cols = [], []                               # plant capacity: sum of the plant's flows <= capacity
    for e, ed in enumerate(edges):
        if ed["kind"] == "plant":
            rows += [PLANTS.index(ed["plant"]) * H + t]; cols += [e * H + t]
    rows, cols = np.concatenate(rows), np.concatenate(cols)
    A_cap = sparse.csr_matrix((np.ones(len(rows)), (rows, cols)), shape=(P * H, n))

    rows, cols, vals = [], [], []                     # -(flow + slack) <= -min flow;  -(V + slack) <= -min volume
    for f, rv in enumerate(MINFLOW_RIVERS):
        for e, ed in enumerate(edges):
            if (ed["collector"] if rv == COLLECTOR else ed["river"] == rv):
                rows += [f * H + t]; cols += [e * H + t]; vals += [-np.ones(H)]
        rows += [f * H + t]; cols += [o_f + f * H + t]; vals += [-np.ones(H)]
    for k, r in enumerate(MINVOL_RES):
        rows += [F * H + k * H + t, F * H + k * H + t]; cols += [o_v + ri[r] * H + t, o_m + k * H + t]
        vals += [-np.ones(H), -np.ones(H)]
    A_con = sparse.csr_matrix((np.concatenate(vals), (np.concatenate(rows), np.concatenate(cols))), shape=((F + MV) * H, n))

    gen = m["generator"]
    plant_of = lambda name: name.rsplit("_G", 1)[0]
    return dict(res=res, ri=ri, edges=edges, A_eq=A_eq, A_cap=A_cap, A_con=A_con, n=n, offsets=(o_v, o_f, o_m),
                plant_dest={p: dest_of(p) for p in PLANTS},
                p_max={p: sum(g["p_max"] for nm, g in gen.items() if plant_of(nm) == p) for p in PLANTS},
                q_max={p: sum(max(max(c["x"]) for c in g["turb_eff_curves"]) for nm, g in gen.items() if plant_of(nm) == p)
                       for p in PLANTS},
                outlet={p: m["plant"][p]["outlet_line"] for p in PLANTS},
                vol_head={r: (np.array(m["reservoir"][r]["vol_head"]["x"]), np.array(m["reservoir"][r]["vol_head"]["y"]))
                          for r in res},
                max_vol=np.array([m["reservoir"][r]["max_vol"] for r in res]))


def solve_case(M, price, inflow_h, rvest_inflow, wv_end, v0, minflow, minvol):
    """One case at all scales -> flow share (scales, H, plants), margin in EUR/MWh (scales, H, plants), status."""
    res, ri, edges = M["res"], M["ri"], M["edges"]
    R, E, P, n = len(res), len(edges), len(PLANTS), M["n"]
    o_v, o_f, o_m = M["offsets"]
    level = np.array([np.interp(v0[i], *M["vol_head"][r]) for i, r in enumerate(res)])
    k_edge = np.zeros(E)                              # MWh per (m³/s * hour), from the head at the start level
    for e, ed in enumerate(edges):
        if ed["kind"] == "plant":
            k_edge[e] = 0.00981 * ETA * max(level[ri[ed["src"]]] - M["outlet"][ed["plant"]], 1.0)
    k_plant = {p: max(k_edge[e] for e, ed in enumerate(edges) if ed["plant"] == p) for p in PLANTS}
    k_res = {r: next((k_edge[e] for e, ed in enumerate(edges) if ed["plant"] == GROUP_PLANT[r] and ed["src"] == r),
                     k_plant[GROUP_PLANT[r]]) for r in res}
    cap = np.array([min(M["q_max"][p], M["p_max"][p] / k_plant[p]) for p in PLANTS])

    b_eq = (M3S_H * inflow_h.T).ravel().copy()
    b_eq[np.arange(R) * H] += v0
    mf = minflow.copy()
    mf[:, MINFLOW_RIVERS.index(COLLECTOR)] -= rvest_inflow       # the river's own inflow already counts
    b_ub = np.concatenate([np.repeat(cap, H), -np.maximum(mf, 0).T.ravel(), -minvol.T.ravel()])
    A_ub = sparse.vstack([M["A_cap"], M["A_con"]]).tocsr()
    bounds = np.zeros((n, 2))
    bounds[:, 1] = np.inf
    for e, ed in enumerate(edges):
        if ed["ub"] is not None:
            bounds[e * H:(e + 1) * H, 1] = ed["ub"]
    for i in range(R):
        bounds[o_v + i * H:o_v + (i + 1) * H, 1] = max(M["max_vol"][i], v0[i])

    frac = np.full((len(SCALES), H, P), np.nan, dtype=np.float32)
    margin = np.full_like(frac, np.nan)
    status = np.zeros(len(SCALES), dtype=np.int8)
    for s_i, s in enumerate(SCALES):
        V = {}
        def value(r):     # whole-path value of 1 m³/s-hour stored in r: own water value x own plant + downstream
            if r is None:
                return 0.0
            if r not in V:
                V[r] = s * wv_end[ri[r]] * k_res[r] + value(M["plant_dest"][GROUP_PLANT[r]])
            return V[r]
        c = np.zeros(n)
        for e, ed in enumerate(edges):
            c[e * H:(e + 1) * H] = ed["cost"] + value(ed["src"]) - value(ed["dst"]) - (price * k_edge[e] if ed["kind"] == "plant" else 0.0)
        c[o_f:o_m], c[o_m:] = MINFLOW_PENALTY, MINVOL_PENALTY
        out = linprog(c, A_ub=A_ub, b_ub=b_ub, A_eq=M["A_eq"], b_eq=b_eq, bounds=bounds, method="highs")
        status[s_i] = out.status
        if out.status != 0:
            continue
        # reduced cost without the capacity rows: < 0 means the plant would like to run more
        rc = c - M["A_eq"].T @ out.eqlin.marginals - M["A_con"].T @ out.ineqlin.marginals[P * H:]
        for p_i, p in enumerate(PLANTS):
            pe = [e for e, ed in enumerate(edges) if ed["plant"] == p]
            frac[s_i, :, p_i] = sum(out.x[e * H:(e + 1) * H] for e in pe) / cap[p_i]
            margin[s_i, :, p_i] = np.max([-rc[e * H:(e + 1) * H] / k_edge[e] for e in pe], axis=0)
    return frac, margin, status


def run_lp(starts, M):
    """Solve the LP for every case. Returns frac and margin of shape (scales, cases, 168, plants)."""
    starts = pd.DatetimeIndex(starts)
    price = day_ahead_price["price"]
    res = M["res"]
    frac = np.zeros((len(SCALES), len(starts), H, len(PLANTS)), dtype=np.float32)
    margin = np.zeros_like(frac)
    bad, t0 = 0, time.time()
    for i, s in enumerate(starts):
        hours = pd.date_range(s, periods=H, freq="h")
        f, mg, st = solve_case(M,
                               price.reindex(hours).ffill().bfill().to_numpy(),
                               inflow.reindex(hours)[res].fillna(0).to_numpy(),
                               inflow.reindex(hours)[COLLECTOR].fillna(0).to_numpy(),
                               water_value.loc[s + pd.Timedelta(days=7), res].to_numpy(dtype=float),
                               volume.loc[s, res].to_numpy(dtype=float),
                               min_flow.reindex(hours)[MINFLOW_RIVERS].fillna(0).to_numpy(),
                               min_volume.reindex(hours)[MINVOL_RES].fillna(0).to_numpy())
        frac[:, i], margin[:, i], bad = f, mg, bad + int((st != 0).any())
        if (i + 1) % 500 == 0 or i + 1 == len(starts):
            print(f"{i + 1}/{len(starts)} cases, {time.time() - t0:.0f}s, not solved: {bad}")
    return {"frac": frac, "margin": margin}

In [ ]:
lp_model = build_model(topology)
lp_tr = run_lp(unit_commitment_decisions.index, lp_model)   # 2,922 training cases
lp_te = run_lp(test_cases["starttime"], lp_model)            # 725 test cases

In [ ]:
# LP features per unit (from its plant) and for the whole system; rows ordered (case, hour) like SYS
lp_scales = [f"{s:.1f}".replace(".", "") for s in SCALES]       # "09", "10", "11"

def lp_unit_features(lp, unit):
    p = PLANTS.index(unit.rsplit("_G", 1)[0])
    frac, margin = lp["frac"][..., p], lp["margin"][..., p]      # (scales, cases, 168)
    cols = {}
    for s_i, s in enumerate(lp_scales):
        cols[f"lp_frac_{s}"] = frac[s_i].ravel()
        cols[f"lp_margin_{s}"] = margin[s_i].ravel()
    f, m = frac[1], margin[1]                                    # scale 1.0
    cols["lp_frac_roll5"] = rolling_within_case(f, 5).ravel()
    cols["lp_margin_roll5"] = rolling_within_case(m, 5).ravel()
    cols["lp_margin_prev"] = np.concatenate([m[:, :1], m[:, :-1]], axis=1).ravel()   # one hour before
    cols["lp_margin_next"] = np.concatenate([m[:, 1:], m[:, -1:]], axis=1).ravel()   # one hour after
    cols["lp_frac_week"] = np.repeat(f.mean(axis=1), 168)
    cols["lp_margin_week"] = np.repeat(m.mean(axis=1), 168)
    return pd.DataFrame(cols).astype("float32")

def lp_system(lp):
    m = lp["margin"][1]                                          # margin of all 9 plants at scale 1.0
    return pd.DataFrame(m.reshape(-1, m.shape[-1]), columns=[f"lp_sys_margin_{p}" for p in PLANTS]).astype("float32")

LPS, LPS_test = lp_system(lp_tr), lp_system(lp_te)

def unit_table(u, test=False):
    """All features of one unit: its own 43, the 68 system features and the 21 LP features."""
    Xs, S, L, lp = (X_test, SYS_test, LPS_test, lp_te) if test else (X, SYS, LPS, lp_tr)
    base = Xs.loc[(Xs["unit"] == u).to_numpy(), unit_feats].reset_index(drop=True)
    return pd.concat([base, S, L, lp_unit_features(lp, u)], axis=1)

print(unit_table(units[0]).shape, unit_table(units[0], test=True).shape)   # (490896, 132) (121800, 132)

## One XGBoost model per unit (3 seeds)

Settings from the validation in the report (`02_modelling.ipynb`, rounds 2 and 5): 1,000 trees, learning rate 0.03,
depth 6, `min_child_weight` 20, row subsample 0.8, column subsample 0.6, `reg_lambda` 10. Trained on all 2,922 labelled
cases; the test prediction is the average over seeds 42, 43 and 44. About 20–30 minutes.

In [ ]:
unit_params = {"n_estimators": 1000, "learning_rate": 0.03, "max_depth": 6, "min_child_weight": 20,
               "subsample": 0.8, "colsample_bytree": 0.6, "reg_lambda": 10, "max_bin": 256,
               "tree_method": "hist", "n_jobs": -1}
SEEDS = (42, 43, 44)

test_unit = X_test["unit"].to_numpy()
n_test = len(test_cases)
p = np.zeros(len(X_test))
for u in units:
    t0 = time.time()
    Z, Zt, yu = unit_table(u), unit_table(u, test=True), y[(X["unit"] == u).to_numpy()]
    p[test_unit == u] = np.mean([xgb.XGBClassifier(**unit_params, random_state=sd).fit(Z, yu).predict_proba(Zt)[:, 1]
                                 for sd in SEEDS], axis=0)
    print(f"{u}: {time.time() - t0:.0f}s")
P_trees = p.reshape(len(units), n_test, 168).transpose(1, 0, 2).reshape(n_test, -1)   # (test case, unit*hour) = Kaggle layout

## Neural network with system features (3 seeds)

A 1D CNN over the 168 hours, shared by all 14 units (unit embedding, dilated residual blocks with GroupNorm, the mean
of all units and of the whole week as context). The 68 system values per hour are projected and added to every unit.
Inputs: the 28 unit features of the earlier models (`v6` set without `unit`), standardised with the training cases
and clipped at ±5. Trained on all cases for 15 epochs (median best epoch in validation), seeds 42, 43, 44.

The network is trained in a **separate Python process** (`%%script`): on macOS, PyTorch and XGBoost ship different
OpenMP runtimes that can crash a single process. The inputs are passed through `.npy` files. Device: Apple GPU (`mps`)
if available, else CUDA, else CPU (CPU is slower, about 1–2 hours).

In [ ]:
# Inputs for the network: (case, unit, hour, feature) arrays + system features (case, hour, 68), standardised
nn_features = [c for c in v6_features if c != "unit"]                       # 28

def to_weeks(X_rows, n):
    A = X_rows[nn_features].to_numpy(dtype=np.float32).reshape(len(units), n, 168, len(nn_features))
    return A.transpose(1, 0, 2, 3)

def sys_array(S_rows, n):
    """Signed log on the price/water-value columns so the 2022 price spikes do not dominate."""
    D = S_rows.copy()
    for c in D.columns:
        if c.startswith(("sys_margin_", "sys_wv_end_")):
            v = D[c].to_numpy()
            D[c] = np.sign(v) * np.log1p(np.abs(v))
    return D.to_numpy(dtype=np.float32).reshape(n, 168, D.shape[1])

def standardize(train, test, axes):
    mean = train.mean(axis=axes, keepdims=True)
    std = train.std(axis=axes, keepdims=True) + 1e-6
    return (np.clip((train - mean) / std, -5, 5).astype(np.float32), np.clip((test - mean) / std, -5, 5).astype(np.float32))

A_all, A_test_s = standardize(to_weeks(X, n_cases), to_weeks(X_test, n_test), axes=(0, 1, 2))
S_all, S_test_s = standardize(sys_array(SYS, n_cases), sys_array(SYS_test, n_test), axes=(0, 1))
for name, arr in [("A_all", A_all), ("A_test", A_test_s), ("S_all", S_all), ("S_test", S_test_s), ("Y", Y.astype(np.float32))]:
    np.save(f"nn_in_{name}.npy", arr)
print(A_all.shape, A_test_s.shape, S_all.shape, S_test_s.shape)

In [ ]:
%%script "{sys.executable}"
# --- runs in a separate Python process (see above); reads nn_in_*.npy, writes nn_sys_test.npy ---
import time
import numpy as np
import torch
from torch import nn

EPOCHS, SEEDS, BATCH, LR = 15, (42, 43, 44), 32, 3e-4
A, S, Yw = (np.load(f"nn_in_{k}.npy") for k in ["A_all", "S_all", "Y"])
A_test, S_test = np.load("nn_in_A_test.npy"), np.load("nn_in_S_test.npy")
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("Device:", device, flush=True)

class SysUnitCNN(nn.Module):
    def __init__(self, n_features, n_sys, n_units=14, emb_dim=8, hidden=64, dilations=(1, 2, 4, 8, 16, 32, 64, 1), dropout=0.1):
        super().__init__()
        self.unit_emb = nn.Embedding(n_units, emb_dim)
        self.inp = nn.Conv1d(n_features + emb_dim, hidden, kernel_size=1)
        self.sys_proj = nn.Conv1d(n_sys, hidden, kernel_size=1)
        self.convs = nn.ModuleList([nn.Conv1d(3 * hidden, hidden, kernel_size=3, padding=d, dilation=d) for d in dilations])
        self.norms = nn.ModuleList([nn.GroupNorm(8, hidden) for _ in dilations])
        self.drop = nn.Dropout(dropout)
        self.out = nn.Conv1d(hidden, 1, kernel_size=1)

    def forward(self, x, s):                   # x: (B, U, H, F), s: (B, H, S)
        B, U, H, F = x.shape
        emb = self.unit_emb.weight[None, :, None, :].expand(B, U, H, -1)
        x = torch.cat([x, emb], dim=-1).reshape(B * U, H, -1).transpose(1, 2)
        s = self.sys_proj(s.transpose(1, 2))[:, None].expand(B, U, -1, H).reshape(B * U, -1, H)
        h = torch.relu(self.inp(x) + s)
        for conv, norm in zip(self.convs, self.norms):
            ctx = h.reshape(B, U, -1, H).mean(dim=1, keepdim=True).expand(B, U, -1, H).reshape(B * U, -1, H)   # all units
            week = h.mean(dim=2, keepdim=True).expand(-1, -1, H)                                            # whole week
            h = h + self.drop(torch.relu(norm(conv(torch.cat([h, ctx, week], dim=1)))))
        return self.out(h).reshape(B, U, H)

@torch.no_grad()
def predict(model, X, S_, batch_size=128):
    model.eval()
    return torch.cat([torch.sigmoid(model(X[i:i + batch_size].to(device), S_[i:i + batch_size].to(device))).cpu()
                      for i in range(0, len(X), batch_size)]).numpy()

X_tr, S_tr, Y_tr = torch.from_numpy(A), torch.from_numpy(S), torch.from_numpy(Yw)
preds = []
for seed in SEEDS:
    torch.manual_seed(seed)
    model = SysUnitCNN(A.shape[-1], S.shape[-1]).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    loss_fn = nn.BCEWithLogitsLoss()
    for epoch in range(1, EPOCHS + 1):
        t0, total = time.time(), 0.0
        model.train()
        order = torch.randperm(len(X_tr))
        for i in range(0, len(order), BATCH):
            idx = order[i:i + BATCH]
            opt.zero_grad()
            loss = loss_fn(model(X_tr[idx].to(device), S_tr[idx].to(device)), Y_tr[idx].to(device))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            total += loss.item() * len(idx)
        print(f"seed {seed} | epoch {epoch:2d} | loss {total / len(X_tr):.4f} | {time.time() - t0:.0f}s", flush=True)
    preds.append(predict(model, torch.from_numpy(A_test), torch.from_numpy(S_test)).reshape(len(A_test), -1))
np.save("nn_sys_test.npy", np.mean(preds, axis=0))
print("saved nn_sys_test.npy", np.mean(preds, axis=0).shape)

## Blend and submission

In [ ]:
# Rank blend (weights from the validation sweep in the report): trees 0.95, NN 0.05
P_nn = np.load("nn_sys_test.npy")
assert P_nn.shape == P_trees.shape
rr = lambda q: rankdata(np.ravel(q)) / np.size(q)
print("Rank correlation trees vs NN on test:", round(np.corrcoef(rr(P_trees), rr(P_nn))[0, 1], 3))
blend = 0.95 * rr(P_trees) + 0.05 * rr(P_nn)

submission = pd.DataFrame(blend.reshape(P_trees.shape), columns=label_cols)
submission.insert(0, "Run No", test_cases["Run No"].to_numpy())
submission.to_csv(OUTPUT, index=False)
print("saved", OUTPUT, submission.shape)   # (725, 2353)